# broadcast-initial-weights — ex2: broadcast a full state_dict — params AND BN buffers

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcast-initial-weights`. Running the final beacon cell reports progress against the `Distributed: broadcast initial weights` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: broadcast initial weights` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcast-initial-weights`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcast-initial-weights"
DD_SUBTOPIC = "Distributed: broadcast initial weights"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting a full `state_dict` — params + buffers

Ex1 broadcast each `nn.Parameter.data`. That misses two things in a real DDP init:

1. **BatchNorm running stats** (`running_mean`, `running_var`, `num_batches_tracked`) are NOT parameters — they live in `model.buffers()`. Ranks must agree on them too, or the first eval batch sees rank-specific BN stats.
2. **Non-float buffers** (LongTensors, mask buffers, etc.) — must still be in sync.

Canonical pattern: iterate `model.state_dict().values()` (which yields params + persistent buffers) and broadcast each tensor:

```python
for tensor in model.state_dict().values():
    dist.broadcast(tensor, src=0)
```

**Why state_dict, not parameters + buffers.** `state_dict()` returns the canonical superset in a deterministic order — same order on every rank because every rank has the same model graph. Zipping per-rank lists from `parameters()` and `buffers()` separately is more typing for the same answer.

**state_dict tensors are VIEWS into the underlying storage.** Mutating them in-place (which is what `dist.broadcast` does) updates the live model. No `model.load_state_dict()` call needed afterward.

### Exercise 2 — broadcast a full state_dict — params AND BN buffers

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `dist.broadcast(tensor, src=0)` over every tensor in `model.state_dict().values()` so every rank's parameters AND BatchNorm buffers (`running_mean`, `running_var`) all mirror rank 0 at the start of training.
> Keywords: broadcast, state_dict, BN-buffers, running_mean, DDP-init
> ```

**KCs targeted:** `broadcast-state-dict-values`, `buffers-need-sync-too`

Implement `ex2_broadcast_state_dict(rank, world_size, dist_module, model)`. The full-state DDP init pattern:

1. Iterate `model.state_dict().values()`. Order is deterministic across ranks because every rank has the same model graph.
2. For each yielded tensor, call `dist_module.broadcast(tensor, src=0)`. This mutates the underlying storage in-place; no `load_state_dict` call needed.
3. Return the count of tensors broadcast (used by tests to verify both params AND buffers were visited).

Signature note: `dist_module` is injected (a mocked `dist` on CPU). All calls go via `dist_module.broadcast(...)`, not the global `dist.broadcast`.

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module` with rank-specific initial params AND buffers.
Output: `int` — count of tensors broadcast (i.e. `len(model.state_dict())`).

In [ ]:
def ex2_broadcast_state_dict(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    """Broadcast every tensor in model.state_dict() from rank 0. Return tensor count."""
    raise NotImplementedError()


def _test_ex2():

    import threading
    import types as _types
    import torch as _t_for_fake

    class _FakeReduceOp:
        SUM = 'SUM'
        MAX = 'MAX'
        MIN = 'MIN'
        PRODUCT = 'PROD'

    class _FakeWorld:
        """Shared state across `world_size` rank-threads."""
        def __init__(self, world_size):
            self.world_size = world_size
            self.barrier = threading.Barrier(world_size)
            self.lock = threading.Lock()
            self.scratch = {}
            self.tls = threading.local()
            self.results = [None] * world_size
            # send/recv mailbox keyed by (src, dst)
            self.mailbox = {}
            self.mailbox_cv = threading.Condition(self.lock)
        def all_reduce(self, tensor, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('ar', [None] * self.world_size)
                self.scratch['ar'][rank] = tensor.detach().clone()
            self.barrier.wait()
            bag = self.scratch['ar']
            if op == 'SUM':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced + x
            elif op == 'MAX':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.maximum(reduced, x)
            elif op == 'MIN':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = _t_for_fake.minimum(reduced, x)
            elif op == 'PROD':
                reduced = bag[0].clone()
                for x in bag[1:]:
                    reduced = reduced * x
            else:
                raise ValueError(f'unknown fake op {op!r}')
            tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('ar', None)
            self.barrier.wait()
        def reduce(self, tensor, dst, op='SUM'):
            rank = self.tls.rank
            self.barrier.wait()
            with self.lock:
                self.scratch.setdefault('rd', [None] * self.world_size)
                self.scratch['rd'][rank] = tensor.detach().clone()
            self.barrier.wait()
            if rank == dst:
                bag = self.scratch['rd']
                if op == 'SUM':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced + x
                elif op == 'MAX':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.maximum(reduced, x)
                elif op == 'MIN':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = _t_for_fake.minimum(reduced, x)
                elif op == 'PROD':
                    reduced = bag[0].clone()
                    for x in bag[1:]:
                        reduced = reduced * x
                else:
                    raise ValueError(f'unknown fake op {op!r}')
                tensor.copy_(reduced)
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('rd', None)
            self.barrier.wait()
        def broadcast(self, tensor, src):
            rank = self.tls.rank
            self.barrier.wait()
            if rank == src:
                with self.lock:
                    self.scratch['bc'] = tensor.detach().clone()
            self.barrier.wait()
            if rank != src:
                tensor.copy_(self.scratch['bc'])
            self.barrier.wait()
            if rank == 0:
                self.scratch.pop('bc', None)
            self.barrier.wait()
        def barrier_op(self):
            self.barrier.wait()
        def send(self, tensor, dst):
            rank = self.tls.rank
            with self.mailbox_cv:
                self.mailbox.setdefault((rank, dst), []).append(tensor.detach().clone())
                self.mailbox_cv.notify_all()
        def recv(self, tensor, src):
            rank = self.tls.rank
            with self.mailbox_cv:
                while not self.mailbox.get((src, rank)):
                    self.mailbox_cv.wait(timeout=10)
                payload = self.mailbox[(src, rank)].pop(0)
            tensor.copy_(payload)

    def _run_fake_world(worker_fn, world_size, *extra_args, timeout=30):
        world = _FakeWorld(world_size)
        errors = [None] * world_size
        def _runner(rank):
            world.tls.rank = rank
            fake_dist = _types.SimpleNamespace()
            fake_dist.ReduceOp = _FakeReduceOp
            fake_dist.all_reduce = lambda tensor, op='SUM': world.all_reduce(tensor, op)
            fake_dist.reduce = lambda tensor, dst, op='SUM': world.reduce(tensor, dst, op)
            fake_dist.broadcast = lambda tensor, src: world.broadcast(tensor, src)
            fake_dist.barrier = world.barrier_op
            fake_dist.get_rank = lambda: rank
            fake_dist.get_world_size = lambda: world_size
            fake_dist.send = lambda tensor, dst: world.send(tensor, dst)
            fake_dist.recv = lambda tensor, src: world.recv(tensor, src)
            fake_dist.init_process_group = lambda **kw: world.scratch.setdefault('_init_calls', []).append(kw)
            fake_dist.destroy_process_group = lambda: world.scratch.setdefault('_destroy_calls', []).append(rank)
            try:
                worker_fn(rank, world_size, fake_dist, world)
            except BaseException as e:
                import traceback as _tb
                errors[rank] = (e, _tb.format_exc())
        threads = [threading.Thread(target=_runner, args=(r,), daemon=True) for r in range(world_size)]
        for th in threads:
            th.start()
        for th in threads:
            th.join(timeout=timeout)
        for r, err in enumerate(errors):
            if err is not None:
                raise RuntimeError(f'rank {r} failed: {err[0]!r}\n{err[1]}')
        return world


    import torch.nn as nn

    # Model with BOTH params (Linear weight/bias) and buffers (BN running stats).
    class _NetWithBN(nn.Module):
        def __init__(self):
            super().__init__()
            self.lin = nn.Linear(4, 4)
            self.bn = nn.BatchNorm1d(4)

    def _worker(rank, world_size, dist_module, world):
        t.manual_seed(rank + 1)
        model = _NetWithBN()
        # Force per-rank divergence on params AND BN running stats.
        with t.no_grad():
            model.lin.weight.fill_(float(rank + 1))
            model.lin.bias.fill_(float(rank + 1) * 10)
            model.bn.running_mean.fill_(float(rank + 1) * 100)
            model.bn.running_var.fill_(float(rank + 1) * 1000)
        count = ex2_broadcast_state_dict(rank, world_size, dist_module, model)
        world.results[rank] = (count,
                                model.lin.weight.detach().clone(),
                                model.lin.bias.detach().clone(),
                                model.bn.running_mean.detach().clone(),
                                model.bn.running_var.detach().clone())

    w = _run_fake_world(_worker, 3)
    rank0 = w.results[0]
    expected_count = rank0[0]
    # state_dict on this model: lin.weight, lin.bias, bn.weight, bn.bias,
    # bn.running_mean, bn.running_var, bn.num_batches_tracked → at least 5 broadcast.
    assert expected_count >= 5, f'expected at least 5 tensors broadcast, got {expected_count}'

    # Every rank's params + buffers must equal rank 0's.
    for r in range(3):
        cnt, w_lw, w_lb, w_rm, w_rv = w.results[r]
        assert cnt == expected_count, f'rank {r}: count {cnt} vs rank0 {expected_count}'
        assert t.allclose(w_lw, rank0[1]), f'rank {r}: lin.weight diverged from rank 0'
        assert t.allclose(w_lb, rank0[2]), f'rank {r}: lin.bias diverged from rank 0'
        assert t.allclose(w_rm, rank0[3]), f'rank {r}: bn.running_mean diverged from rank 0 (BUFFERS missed!)'
        assert t.allclose(w_rv, rank0[4]), f'rank {r}: bn.running_var diverged'

    # Rank 0's values should equal what we set (1.0 weight, 10.0 bias, 100.0 mean, 1000.0 var).
    assert abs(rank0[1].mean().item() - 1.0) < 1e-5
    assert abs(rank0[2].mean().item() - 10.0) < 1e-5
    assert abs(rank0[3].mean().item() - 100.0) < 1e-5
    assert abs(rank0[4].mean().item() - 1000.0) < 1e-5

    # 2-rank case — degenerate broadcast (one sender, one receiver).
    def _worker2(rank, world_size, dist_module, world):
        model = _NetWithBN()
        with t.no_grad():
            model.lin.weight.fill_(float(rank + 5))
            model.bn.running_mean.fill_(float(rank + 1) * 7.0)
        ex2_broadcast_state_dict(rank, world_size, dist_module, model)
        world.results[rank] = (model.lin.weight.detach().clone(),
                                model.bn.running_mean.detach().clone())

    w2 = _run_fake_world(_worker2, 2)
    # Rank 0 had weight=5.0, mean=7.0. Both ranks should end with those.
    for r in range(2):
        lw, rm = w2.results[r]
        assert abs(lw.mean().item() - 5.0) < 1e-5, f'2-rank: rank {r} lin.weight={lw.mean().item()}'
        assert abs(rm.mean().item() - 7.0) < 1e-5, f'2-rank: rank {r} running_mean={rm.mean().item()}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_broadcast_state_dict(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    count = 0
    for tensor in model.state_dict().values():
        dist_module.broadcast(tensor, src=0)
        count += 1
    return count
```

**Why iterate `state_dict().values()` not `parameters()`.** `parameters()` yields ONLY learnable tensors — Linear weight/bias, Conv weight/bias. It SKIPS BatchNorm running stats, which are persistent buffers, not parameters. Broadcasting params alone leaves a silent bug: per-rank `running_mean` drifts forever.

**`num_batches_tracked` is an `int64` scalar.** Some older backends choke on int broadcast (gloo handles it fine, nccl is finicky). The fake harness above handles all dtypes uniformly via `copy_`.

**Order-determinism.** `state_dict()` returns an `OrderedDict` whose key order is the in-graph traversal order. Every rank has the same model graph, so every rank iterates the same order — no off-by-one or interleaved broadcasts.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()